# NB01: Data Extraction and QC

Extract genome-level CAZy profiles from `kescience_mgnify`, quality-filter, and characterize the dataset.

In [ ]:
from berdl_notebook_utils.setup_spark_session import get_spark_session
import pandas as pd
import numpy as np
import os

spark = get_spark_session()
DATA_DIR = '../data'
FIG_DIR = '../figures'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Extract genome-level CAZy profiles with metadata

In [ ]:
genome_cazy_df = spark.sql("""
    SELECT g.genome_id, g.biome_id, g.species_id, g.lineage,
           g.length, g.n_contigs, g.gc_content,
           g.completeness, g.contamination, g.genome_type,
           g.is_species_rep, g.country, g.continent,
           c.cazy_family, c.gene_count
    FROM kescience_mgnify.genome g
    JOIN kescience_mgnify.genome_cazy c ON g.genome_id = c.genome_id
    WHERE g.completeness >= 50
""")

print(f'Rows after completeness >= 50 filter: {genome_cazy_df.count():,}')
print(f'Distinct genomes: {genome_cazy_df.select("genome_id").distinct().count():,}')
print(f'Distinct species: {genome_cazy_df.select("species_id").distinct().count():,}')

In [ ]:
# Pivot to wide format: one row per genome, columns for each CAZy class
profiles = genome_cazy_df.toPandas()
print(f'Pandas DataFrame shape: {profiles.shape}')
print(f'CAZy classes: {sorted(profiles.cazy_family.unique())}')
profiles.head()

In [ ]:
# Pivot to wide format
meta_cols = ['genome_id', 'biome_id', 'species_id', 'lineage', 'length',
             'n_contigs', 'gc_content', 'completeness', 'contamination',
             'genome_type', 'is_species_rep', 'country', 'continent']

wide = profiles.pivot_table(
    index=meta_cols,
    columns='cazy_family',
    values='gene_count',
    fill_value=0
).reset_index()

wide.columns.name = None
cazy_classes = [c for c in wide.columns if c not in meta_cols]
wide['total_cazy'] = wide[cazy_classes].sum(axis=1)
wide['cazy_density'] = wide['total_cazy'] / (wide['length'] / 1e6)  # per Mbp

print(f'Wide format: {wide.shape[0]:,} genomes x {len(cazy_classes)} CAZy classes')
print(f'CAZy classes: {cazy_classes}')
wide.describe()

## 2. Biome distribution

In [ ]:
biome_summary = wide.groupby('biome_id').agg(
    n_genomes=('genome_id', 'count'),
    n_species=('species_id', 'nunique'),
    mean_total_cazy=('total_cazy', 'mean'),
    mean_GH=('GH', 'mean'),
    mean_GT=('GT', 'mean'),
    mean_length_Mbp=('length', lambda x: (x / 1e6).mean()),
    mean_completeness=('completeness', 'mean'),
    pct_MAG=('genome_type', lambda x: (x == 'MAG').mean() * 100)
).sort_values('n_genomes', ascending=False)

biome_summary = biome_summary.round(1)
print(biome_summary.to_string())
biome_summary.to_csv(f'{DATA_DIR}/biome_summary.tsv', sep='\t')

## 3. Phylum distribution

In [ ]:
# Extract phylum from GTDB lineage
wide['gtdb_phylum'] = wide['lineage'].str.extract(r'p__([^;]+)')

phylum_summary = wide.groupby('gtdb_phylum').agg(
    n_genomes=('genome_id', 'count'),
    mean_total_cazy=('total_cazy', 'mean'),
    mean_GH=('GH', 'mean'),
    mean_GT=('GT', 'mean'),
    mean_cazy_density=('cazy_density', 'mean')
).sort_values('n_genomes', ascending=False).head(20)

print(phylum_summary.round(1).to_string())

## 4. Genome type breakdown (Isolate vs MAG)

In [ ]:
type_summary = wide.groupby('genome_type').agg(
    n_genomes=('genome_id', 'count'),
    mean_total_cazy=('total_cazy', 'mean'),
    mean_GH=('GH', 'mean'),
    mean_GT=('GT', 'mean'),
    mean_completeness=('completeness', 'mean'),
    mean_length_Mbp=('length', lambda x: (x / 1e6).mean())
)
print(type_summary.round(1).to_string())

## 5. Extract pangenome stats

In [ ]:
pangenome_df = spark.sql("""
    SELECT species_id, biome_id, genome_count,
           total_genes, core_genes, soft_core_genes,
           shell_genes, cloud_genes
    FROM kescience_mgnify.pangenome_stats
    WHERE genome_count >= 3
""").toPandas()

pangenome_df['cloud_fraction'] = pangenome_df['cloud_genes'] / pangenome_df['total_genes']
pangenome_df['core_fraction'] = pangenome_df['core_genes'] / pangenome_df['total_genes']

print(f'Species with pangenome stats (>= 3 genomes): {len(pangenome_df):,}')
print(pangenome_df.describe().round(2))
pangenome_df.to_csv(f'{DATA_DIR}/pangenome_stats.tsv', sep='\t', index=False)

## 6. Save genome-level profiles

In [ ]:
wide.to_csv(f'{DATA_DIR}/genome_cazy_profiles.tsv', sep='\t', index=False)
print(f'Saved {len(wide):,} genome profiles to {DATA_DIR}/genome_cazy_profiles.tsv')
print(f'Columns: {list(wide.columns)}')

## Summary

- Extracted genome-level CAZy profiles with metadata and QC filters
- Characterized biome, phylum, and genome type distributions
- Extracted pangenome stats for species with >= 3 genomes
- Saved data for downstream analysis